# L-system playground Testbench

In [64]:
#automatically reload all modules, if you edit something in the .py files
%load_ext autoreload
%autoreload 2
from lsys import derive, interpret, plot, summary, parse_rules

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Classic bracketed plant
`X` is a growth point that draws nothing; `F -> FF` doubles every stem segment each step (older parts get longer).
Source: *The Algorithmic Beauty of Plants*, Fig. 1.24 (f).

In [163]:
gens = derive("X", '''
X -> F() [+//(30)X ]F()[-/(60)X]F()[X] 
F -> F() F(0.6)
''', steps=6)
print(summary(gens[-1]))
plot(interpret(gens[-1], angle=20), "bracketed plant")

F: 1629, [: 1092, /: 1092, ]: 1092, X: 729, +: 364, -: 364


## 2. Stochastic and 3D
Two rules for `B` with weights: every derivation is a different instance of the same "species". Change the seed.
`&` pitches down, `/` rolls around the stem (137.5 deg = golden angle: branches spiral around the stem like real phyllotaxis).

In [170]:
gens = derive("B", '''
B -> F [ &(35) B ] /(137.5) F B @ 0.6
B -> F [ &(35) B ] /(137.5) [ &(35) B ] F B @ 0.25
B -> F B @ 0.15
F -> F F @ 0.5
F -> F @ 0.5
''', steps=10, seed=1)
print(summary(gens[-1]))
plot(interpret(gens[-1]), "stochastic bush", mark_styles={"B": dict(color="limegreen", size=3)})

F: 6471, B: 2194, [: 2193, &: 2193, ]: 2193, /: 1694


## 3. Tip organs (apex that matures)
`A(s, o)`: a tip on an axis of branching order `o` that has grown `s` so far. While it's below its axis' length limit it adds
an internode behind itself, sometimes leaving a new branch tip (order + 1). At the limit it turns into an organ `O`.
Shorter limits for higher orders (`ratio**o`) shape the crown.

In [164]:
rules = '''
A(s, o) : s < maxLen * ratio**o and o < maxOrder -> F(dl) [ &(40) A(0, o+1) ] /(137.5) A(s+dl, o) @ pBranch
A(s, o) : s < maxLen * ratio**o -> F(dl) A(s+dl, o) @ 1 - pBranch
A(s, o) : s >= maxLen * ratio**o -> O(o)
'''
constants = dict(maxLen=4, ratio=0.6, maxOrder=3, dl=0.25, pBranch=0.3)
gens = derive("A(0, 0)", rules, steps=30, constants=constants, seed=0)
print(summary(gens[-1]))
plot(interpret(gens[-1]), "tip organs", mark_styles={"A": dict(color="yellowgreen", size=3), "O": dict(color="green", size=7)})

F: 99, [: 15, &: 15, ]: 15, /: 15, O: 13, A: 3


### Growing step by step
Each generation is kept in `gens`: look at any step, or print its modules to see what the rules produced.

In [166]:
print(gens[4])
plot(interpret(gens[12]), "step 12", mark_styles={"A": dict(color="yellowgreen", size=3), "O": dict(color="green", size=7)})

[F(0.25), F(0.25), F(0.25), F(0.25), [, &(40), A(0, 1), ], /(137.5), A(1, 0)]


## Your experiments